# Pipeline + evaluate2 통합: Colab GPU 학습 → URMP 평가 → W&B 실시간 연동

이 노트북은 기존 `pipeline(2).ipynb`의 데이터 다운로드/체크포인트/W&B 구조와
`evaluate2`에서 하던 X-UMX + URMP SDR 평가 목적을 하나로 합친 버전입니다.

핵심 흐름:

1. GitHub에서 `ensemble_banchmark_pipeline` 코드 다운로드
2. GCS prepared 데이터 → **Colab 로컬 SSD `/content/ensemble-data`** 직접 다운로드
3. GPU에서 SynthSOD train split으로 4-family X-UMX 학습
4. W&B에 batch 단위 loss 실시간 기록
5. 매 epoch 끝에 고정 URMP diagnostic scene SDR 기록
6. 학습 종료 후 전체 validation(SynthSOD + URMP) 공식 SDR 기록
7. checkpoint + config + 평가 report를 W&B Artifact로 저장

> `diagnostic/urmp_sdr_db`는 학습 추세를 보기 위한 부분 점수입니다.
> 최종 비교에는 `eval/urmp_sdr_db`와 `eval/overall_sdr_db`를 사용하세요.


## 1번 셀 — GitHub 코드 다운로드 + 의존성 설치
Colab Secrets에 비공개 저장소라면 `GITHUB_TOKEN`, W&B는 `WANDB_API_KEY`를 등록하세요.


In [ ]:
REPOSITORY = "c-peace/ensemble_banchmark_pipeline"
REF = "main"
CONFIG_FILE = "configs/experiments/baseline.yaml"

import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path


def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


CODE_ROOT = Path("/content/ensemble-code")

if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPOSITORY):
    raise ValueError("REPOSITORY는 owner/repository 형식이어야 합니다.")

url = f"https://github.com/{REPOSITORY}.git"

with tempfile.TemporaryDirectory() as temporary:
    askpass = Path(temporary) / "askpass.py"
    askpass.write_text(
        "#!/usr/bin/env python3\n"
        "import os, sys\n"
        "print('x-access-token' if 'username' in sys.argv[1].lower() "
        "else os.environ.get('GITHUB_TOKEN', ''))\n"
    )
    askpass.chmod(0o700)

    git_env = dict(
        os.environ,
        GIT_TERMINAL_PROMPT="0",
        GIT_ASKPASS=str(askpass),
        GITHUB_TOKEN=secret("GITHUB_TOKEN") or "",
    )

    def git(*args):
        result = subprocess.run(
            ["git", *args],
            env=git_env,
            capture_output=True,
            text=True,
        )
        if result.returncode:
            raise RuntimeError(
                "GitHub 다운로드 실패:\n"
                + result.stderr[-2000:]
            )
        return result.stdout.strip()

    if CODE_ROOT.exists():
        shutil.rmtree(CODE_ROOT)

    checkout = Path(temporary) / "checkout"
    git("init", "--quiet", str(checkout))
    git("-C", str(checkout), "remote", "add", "origin", url)
    git("-C", str(checkout), "fetch", "--quiet", "--depth", "1", "origin", REF)
    git("-C", str(checkout), "checkout", "--quiet", "--detach", "FETCH_HEAD")
    shutil.move(str(checkout), str(CODE_ROOT))

    CODE_COMMIT = git("-C", str(CODE_ROOT), "rev-parse", "HEAD")

os.chdir(CODE_ROOT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"],
    check=True,
)

print("✅ 코드 준비 완료")
print("commit:", CODE_COMMIT)
print("root  :", CODE_ROOT)


## 2번 셀 — GPU 확인 + Colab 학습 프로필

`colab_safe`가 기본입니다.  
무료/일반 Colab에서 메모리 문제가 있으면 `smoke`로 연결부터 확인하세요.

`steps_per_shard`가 숫자면 전체 epoch 대신 그 batch 수만큼만 학습합니다.
성능을 더 올리고 싶으면 `colab_long` 또는 `steps_per_shard=None`으로 확장합니다.


In [ ]:
# 1번 셀에서 등록한 competition-kit 경로를 다시 확인합니다.
import sys
from pathlib import Path

KIT_ROOT = Path("/content/ensemble-code/competition-kit")

if not KIT_ROOT.is_dir():
    raise RuntimeError(
        "1번 셀이 정상적으로 실행되지 않았습니다. "
        f"폴더 없음: {KIT_ROOT}"
    )

kit_str = str(KIT_ROOT)
if kit_str not in sys.path:
    sys.path.insert(0, kit_str)

import copy
import psutil
import torch

from ensemble_benchmark.config import load_config

base_config = load_config(CODE_ROOT / CONFIG_FILE)

PROFILE = "colab_safe"

PROFILES = {
    "smoke": {
        "epochs": 1,
        "tiny": True,
        "steps_per_shard": 2,
        "batch_size": 1,
        "samples_per_track": 1,
        "segment_seconds": 2.0,
        "save_every": 1,
    },
    "colab_safe": {
        "epochs": 3,
        "tiny": False,
        "steps_per_shard": 100,
        "batch_size": 1,
        "samples_per_track": 4,
        "segment_seconds": 3.0,
        "save_every": 25,
    },
    "colab_long": {
        "epochs": 5,
        "tiny": False,
        "steps_per_shard": 300,
        "batch_size": 1,
        "samples_per_track": 8,
        "segment_seconds": 4.0,
        "save_every": 50,
    },
}

if PROFILE not in PROFILES:
    raise ValueError(PROFILE)

config = copy.deepcopy(base_config)

# 실제 데이터 학습. smoke profile일 때만 tiny model을 쓰되
# 데이터 release 자체는 full prepared split을 사용합니다.
config["data"]["smoke"] = False
config["training"].update(PROFILES[PROFILE])

# Colab에서 비교적 안전한 기본 optimizer 설정 유지
config["training"]["learning_rate"] = 1e-4
config["training"]["weight_decay"] = 1e-5
config["training"]["max_grad_norm"] = 5.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print("Colab runtime")
print("=" * 60)
print("PROFILE:", PROFILE)
print("DEVICE :", DEVICE)

ram = psutil.virtual_memory()
print(f"RAM total     : {ram.total / 1024**3:.2f} GB")
print(f"RAM available : {ram.available / 1024**3:.2f} GB")

if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print("GPU :", torch.cuda.get_device_name(0))
    print(f"VRAM: {prop.total_memory / 1024**3:.2f} GB")
else:
    raise RuntimeError(
        "GPU가 없습니다. Colab > 런타임 > 런타임 유형 변경 > GPU를 선택하세요."
    )

print("\nTraining config:")
for k, v in config["training"].items():
    print(f"  {k}: {v}")


## 3번 셀 — GCS prepared 데이터 → Colab 로컬 SSD 직접 다운로드

Google Drive를 거치지 않습니다.

- 다운로드 위치: `/content/ensemble-data`
- 학습: `/content/ensemble-data/train`
- 평가: `/content/ensemble-data/validation`

현재 저장소의 validation selection은 SynthSOD validation과 URMP validation을 함께 포함합니다.


In [ ]:
from google.colab import auth
from ensemble_benchmark.cloud import download_data

auth.authenticate_user(project_id=config["data"]["gcp_project"])

DATA_ROOT = download_data(
    config["data"]["gcs_uri"],
    "/content/ensemble-data",
    project=config["data"]["gcp_project"],
)

TRAIN_ROOT = DATA_ROOT / "train"
VALID_ROOT = DATA_ROOT / "validation"

print("✅ Colab 로컬 SSD 데이터 준비 완료")
print("DATA_ROOT :", DATA_ROOT)
print("TRAIN_ROOT:", TRAIN_ROOT)
print("VALID_ROOT:", VALID_ROOT)

print("\nDisk:")
usage = shutil.disk_usage("/content")
print(f"total: {usage.total/1024**3:.1f} GB")
print(f"used : {usage.used/1024**3:.1f} GB")
print(f"free : {usage.free/1024**3:.1f} GB")


## 4번 셀 — W&B Run 시작

배치마다 아래 값이 실시간으로 올라갑니다.

- `train/loss_mean`
- `train/strings_loss`
- `train/woodwinds_loss`
- `train/brass_loss`
- `train/other_loss`
- 각 family의 STFT / weighted-SDR loss component
- GPU memory

epoch 끝에는 고정 URMP 진단곡의 `diagnostic/urmp_sdr_db`도 기록합니다.


In [ ]:
import uuid
import wandb
import yaml

api_key = secret("WANDB_API_KEY")
if not api_key:
    raise RuntimeError(
        "Colab Secrets에 WANDB_API_KEY를 등록하세요."
    )

wandb.login(key=api_key, relogin=False)

experiment = config["experiment"]

if wandb.run is not None:
    wandb.run.finish()

RUN_ID = (
    f"xumx-{PROFILE}-"
    f"{uuid.uuid4().hex[:8]}"
)

RUN_DIR = Path("/content/ensemble-runs") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

CONFIG_SNAPSHOT = RUN_DIR / "config.yaml"
CONFIG_SNAPSHOT.write_text(
    yaml.safe_dump(
        config,
        allow_unicode=True,
        sort_keys=False,
    )
)

run = wandb.init(
    project=experiment["wandb_project"],
    entity=experiment["wandb_entity"],
    name=RUN_ID,
    id=RUN_ID,
    mode="online",
    dir=str(RUN_DIR),
    config={
        **config,
        "profile": PROFILE,
        "github_repository": REPOSITORY,
        "github_commit": CODE_COMMIT,
        "runtime_device": DEVICE,
    },
)

# W&B x축을 global_step으로 통일
wandb.define_metric("global_step")
wandb.define_metric("train/*", step_metric="global_step")
wandb.define_metric("train_components/*", step_metric="global_step")
wandb.define_metric("system/*", step_metric="global_step")

print("✅ W&B Run 시작")
print("Run :", RUN_ID)
print("URL :", run.url)


## 5번 셀 — release 생성 + epoch별 URMP diagnostic manifest 준비

최종 점수와 혼동하지 않도록 diagnostic은 **고정된 URMP 1곡**만 사용합니다.
매 epoch 같은 곡을 평가하므로 학습 추세 비교에는 사용할 수 있습니다.

최종 공식 점수는 7번 셀의 전체 validation 평가 결과입니다.


In [ ]:
import json
from ensemble_benchmark import runtime

release = runtime.build_local_release(
    DATA_ROOT,
    smoke=False,
    excluded_train_ids=config["data"]["excluded_train_ids"],
)

validation_manifest = json.loads(
    (VALID_ROOT / "manifest.json").read_text()
)

urmp_scenes = [
    s for s in validation_manifest["scenes"]
    if s.get("dataset") == "URMP"
]

if not urmp_scenes:
    raise RuntimeError(
        "prepared validation에 URMP scene이 없습니다."
    )

# 항상 같은 scene으로 추세 비교
DIAG_URMP_SCENES = 1
selected_diag_scenes = urmp_scenes[:DIAG_URMP_SCENES]

DIAG_MANIFEST_ROOT = RUN_DIR / "diagnostic-manifest"
DIAG_MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

diag_manifest = dict(validation_manifest)
diag_manifest["scenes"] = selected_diag_scenes
diag_manifest["complete"] = False

(DIAG_MANIFEST_ROOT / "manifest.json").write_text(
    json.dumps(
        diag_manifest,
        ensure_ascii=False,
        indent=2,
    )
)

print("✅ release:", release["release_id"])
print("train count     :", release["train_count"])
print("validation count:", release["validation_count"])

print("\nEpoch diagnostic URMP scene:")
for scene in selected_diag_scenes:
    print(
        "-",
        scene["scene_id"],
        scene["target_classes"],
    )


## 6번 셀 — GPU 학습 + W&B 실시간 loss + epoch별 URMP SDR

이 셀이 핵심입니다.

- 각 batch loss → 즉시 W&B
- epoch 종료 → checkpoint 저장
- 고정 URMP scene separation → diagnostic SDR → 즉시 W&B
- 다음 epoch은 같은 checkpoint에서 이어서 학습

`diagnostic/urmp_sdr_db`는 부분평가이므로 최종 공식 점수로 사용하지 않습니다.


In [ ]:
import gc
import math
import numpy as np
import torch

from ensemble_benchmark import runtime


def log_training(event):
    """runtime.train_local의 batch callback -> W&B."""
    if event["type"] == "train":
        step = int(event["global_step"])
        losses = event["losses"]

        metrics = {
            "global_step": step,
            "epoch": int(event["epoch"]) + 1,
            "train/loss_mean": float(
                sum(losses.values()) / len(losses)
            ),
            "train/elapsed_seconds": float(
                event.get("elapsed_seconds", 0.0)
            ),
        }

        for family, value in losses.items():
            metrics[f"train/{family}_loss"] = float(value)

        for family, components in event.get(
            "loss_components", {}
        ).items():
            for name, value in components.items():
                metrics[
                    f"train_components/{family}/{name}"
                ] = float(value)

        if torch.cuda.is_available():
            metrics["system/gpu_allocated_gb"] = (
                torch.cuda.memory_allocated() / 1024**3
            )
            metrics["system/gpu_reserved_gb"] = (
                torch.cuda.memory_reserved() / 1024**3
            )
            metrics["system/gpu_max_allocated_gb"] = (
                torch.cuda.max_memory_allocated() / 1024**3
            )

        run.log(metrics, step=step)


def log_diagnostic(report, epoch):
    """부분 URMP report -> W&B."""
    urmp = report.get("datasets", {}).get("URMP", {})
    score = urmp.get("score")

    metrics = {
        "diagnostic/epoch": epoch,
    }

    if score is not None:
        metrics["diagnostic/urmp_sdr_db"] = float(score)

    for inst, value in urmp.get(
        "instruments", {}
    ).items():
        metrics[
            f"diagnostic/urmp/{inst}_sdr_db"
        ] = float(value)

    run.log(metrics)

    if score is not None:
        run.summary[
            "diagnostic/latest_urmp_sdr_db"
        ] = float(score)


CHECKPOINT = None
TOTAL_EPOCHS = int(config["training"]["epochs"])

# epochs만 분리하고 나머지는 동일 설정 사용
train_settings = dict(config["training"])
train_settings.pop("epochs")

for target_epoch in range(1, TOTAL_EPOCHS + 1):

    print("\n" + "=" * 70)
    print(f"🚀 TRAIN -> epoch {target_epoch}/{TOTAL_EPOCHS}")
    print("=" * 70)

    CHECKPOINT = runtime.train_local(
        release,
        DATA_ROOT,
        RUN_DIR,
        device=DEVICE,
        epochs=target_epoch,
        event_callback=log_training,
        **train_settings,
    )

    print("checkpoint:", CHECKPOINT)

    # 학습 모델 객체가 함수 scope에서 해제된 뒤 평가 모델을 다시 로드하므로
    # GPU peak memory를 줄입니다.
    gc.collect()
    torch.cuda.empty_cache()

    print("\n" + "-" * 70)
    print(f"🎯 URMP diagnostic after epoch {target_epoch}")
    print("-" * 70)

    diag_report, diag_report_path = runtime.evaluate(
        release,
        DIAG_MANIFEST_ROOT,
        RUN_DIR,
        device=DEVICE,
        chunk_seconds=10.0,
        allow_partial=True,
        audio_root=VALID_ROOT,
        reuse_predictions=False,
        include_metrics=False,
    )

    log_diagnostic(
        diag_report,
        target_epoch,
    )

    urmp_diag = (
        diag_report
        .get("datasets", {})
        .get("URMP", {})
    )

    print(
        "diagnostic URMP SDR:",
        urmp_diag.get("score"),
        "dB",
    )

    print(
        "instruments:",
        urmp_diag.get("instruments", {}),
    )

    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ 모든 epoch 학습 완료")
print("CHECKPOINT:", CHECKPOINT)


## 7번 셀 — 전체 validation 공식 평가 + W&B SDR 업로드

여기서 나온 값이 최종 비교용입니다.

- `eval/overall_sdr_db`
- `eval/synthsod_sdr_db`
- `eval/urmp_sdr_db`
- `eval/URMP/<instrument>_sdr_db`
- `eval/SynthSOD/<instrument>_sdr_db`

현재 scorer에서 URMP는 12개 악기, SynthSOD는 15개 악기를 평가합니다.


In [ ]:
import numbers
import pandas as pd

evaluation = dict(config["evaluation"])
evaluation.pop("enabled", None)

# 최종 공식 평가이므로 partial 금지
evaluation["allow_partial"] = False
evaluation["max_seconds"] = None
evaluation["reuse_predictions"] = False

report, REPORT_PATH = runtime.evaluate_local(
    release,
    DATA_ROOT,
    RUN_DIR,
    device=DEVICE,
    **evaluation,
)

print("✅ 공식 평가 완료")
print("metric:", report.get("metric"))
print("overall:", report.get("overall"))

official_metrics = {}

if report.get("overall") is not None:
    official_metrics["eval/overall_sdr_db"] = float(
        report["overall"]
    )

for dataset_name, dataset_report in report.get(
    "datasets", {}
).items():

    ds_key = dataset_name.lower()

    if dataset_report.get("score") is not None:
        official_metrics[
            f"eval/{ds_key}_sdr_db"
        ] = float(dataset_report["score"])

    for instrument, value in dataset_report.get(
        "instruments", {}
    ).items():
        official_metrics[
            f"eval/{dataset_name}/{instrument}_sdr_db"
        ] = float(value)

run.log(official_metrics)

for key, value in official_metrics.items():
    run.summary[key] = value

run.summary["evaluation_metric"] = report.get("metric")
run.summary["evaluation_complete"] = bool(
    report.get("complete")
)

print("\nW&B official metrics:")
for key, value in sorted(official_metrics.items()):
    print(f"{key}: {value:.4f} dB")

# 보기 편한 표
rows = []

for dataset_name, dataset_report in report.get(
    "datasets", {}
).items():
    for instrument, value in dataset_report.get(
        "instruments", {}
    ).items():
        rows.append(
            [dataset_name, instrument, float(value)]
        )

score_table = pd.DataFrame(
    rows,
    columns=["dataset", "instrument", "sdr_db"],
)

display(score_table)

wandb_table = wandb.Table(
    dataframe=score_table
)

run.log({
    "eval/instrument_sdr_table": wandb_table
})

print("\nW&B:", run.url)


## 8번 셀 — checkpoint / config / 평가 결과를 W&B Artifact로 저장하고 종료

Colab 런타임을 종료하기 전에 실행하세요.
데이터는 `/content`에서 사라져도 모델과 평가 결과는 W&B에 남습니다.


In [ ]:
import hashlib
import json
from pathlib import Path

if CHECKPOINT is None or not Path(CHECKPOINT).is_file():
    raise FileNotFoundError(
        "checkpoint가 없습니다. 6번 셀을 먼저 실행하세요."
    )

artifact = wandb.Artifact(
    RUN_ID,
    type="model",
    metadata={
        "github_commit": CODE_COMMIT,
        "profile": PROFILE,
        "release_id": release["release_id"],
        "official_overall_sdr_db": report.get("overall"),
        "official_urmp_sdr_db": (
            report.get("datasets", {})
            .get("URMP", {})
            .get("score")
        ),
    },
)

artifact.add_file(
    str(CHECKPOINT),
    name="checkpoint.pt",
)

artifact.add_file(
    str(CONFIG_SNAPSHOT),
    name="config.yaml",
)

if REPORT_PATH is not None and Path(REPORT_PATH).is_file():
    artifact.add_file(
        str(REPORT_PATH),
        name="evaluation.json",
    )

train_log = RUN_DIR / "train-log.jsonl"

if train_log.is_file():
    artifact.add_file(
        str(train_log),
        name="train-log.jsonl",
    )

logged = run.log_artifact(artifact)
logged.wait()

url = run.url

run.finish()

print("✅ W&B 저장 완료")
print(url)


## 사용 권장 순서

처음에는:

```python
PROFILE = "smoke"
```

로 모든 연결을 검증한 뒤:

```python
PROFILE = "colab_safe"
```

로 바꾸세요.

`colab_safe`가 안정적으로 끝나고 SDR이 증가하면 `colab_long`으로 학습량을 늘리면 됩니다.

### W&B에서 봐야 할 값

학습 중:
- `train/loss_mean`
- `train/strings_loss`
- `train/woodwinds_loss`
- `train/brass_loss`
- `train/other_loss`
- `diagnostic/urmp_sdr_db`

최종:
- `eval/urmp_sdr_db`
- `eval/synthsod_sdr_db`
- `eval/overall_sdr_db`

`diagnostic/urmp_sdr_db`는 한 곡 기준 부분 점수이고,
프로젝트 결과표에는 `eval/urmp_sdr_db`를 사용하세요.
